## 0. Prerequisite

Create an environment with uv and install the python binding with 

```bash
uv venv
source .venv/bin/activate
uv pip install -e ./bindings/python
```

In [2]:
import syft_crypto_python as syc

syc.__version__

'0.1.0b5'

In [ ]:
from pprint import pprint

pprint(syc.__all__)

['SyftRecoveryKey',
 'SyftPrivateKeys',
 'SyftPublicKeyBundle',
 'EncryptionRecipient',
 'IdentityMaterial',
 'ParsedEnvelope',
 'compute_key_fingerprint',
 'compute_identity_fingerprint',
 'generate_identity_material',
 'parse_envelope',
 'verify_envelope_signature',
 'encrypt_message',
 'decrypt_message']


## 1. Alice creates the crypto keys

Create everything in one go

In [6]:
mat: syc.IdentityMaterial = syc.generate_identity_material("alice@example.com")

print(f"Identity Fingerprint: {mat.fingerprint}")
print(f"DID (Decentralized Identifier): {mat.did}")
print(f"Recovery key (hex format): {mat.recovery_key_hex}")
print(f"Recovery Phrases: {mat.recovery_key_mnemonic}")
print(f"Private Keys: {mat.key_file}")
print(f"Public Keys Bundle: {mat.public_bundle}")

Identity Fingerprint: 0f604cf2a039ede8f1e85e8a4846e5f813deaa69d7e4c4ec8d1c11e3a63837d0
DID (Decentralized Identifier): did:web:syftbox.net:alice%40example.com
Recovery key (hex format): 88d4-50da-8d4c-1fbb-a2d6-d7a3-ca3e-fef6-dd63-64ec-aaed-0c16-c7cf-cbdc-c850-822a
Recovery Phrases: match penalty cute box sea tape mercy sustain phrase faint sausage unit stomach raven razor frozen army renew view nut smart choose anger fit
Private Keys: b'{\n  "did": "did:web:syftbox.net:alice%40example.com",\n  "format": "syft-private-keys-v1",\n  "identity": "alice@example.com",\n  "identity_fingerprint": "0f604cf2a039ede8f1e85e8a4846e5f813deaa69d7e4c4ec8d1c11e3a63837d0",\n  "private_keys": {\n    "identity_key": {\n      "crv": "Ed25519",\n      "d": "CiEFqjsFy2uDs2O099FSlEgKiTZrLIWEqDY-hklliBNQugUSIIhGNlu9lsdlNWBn24-8I8UjUTi11sIBCEU6Jtl6ca1E",\n      "kid": "identity-key",\n      "kty": "OKP",\n      "use": "sig",\n      "x": "Bao7Bctrg7NjtPfRUpRICok2ayyFhKg2PoZJZYgTULoF"\n    },\n    "pq_prekey": {

Parse a recovery key from a hex string

In [7]:
alice_keys: syc.SyftPrivateKeys = syc.SyftRecoveryKey.from_hex_string(mat.recovery_key_hex).derive_keys()
alice_keys

Create a public bundle from the private keys.

In [8]:
alice_pub: syc.SyftPublicKeyBundle = alice_keys.to_public_bundle()

print(f"Identity Fingerprint: {alice_pub.identity_fingerprint()}")  
print(f"Total Size: {alice_pub.total_size()} (bytes)")

Identity Fingerprint: 0f604cf2a039ede8f1e85e8a4846e5f813deaa69d7e4c4ec8d1c11e3a63837d0
Total Size: 1763 (bytes)


## 2. Bob creates the crypto keys

In [9]:
# Bob
bob_keys = syc.SyftRecoveryKey.generate().derive_keys()
bob_pub = bob_keys.to_public_bundle()

3. Alice encrypts message to Bob

In [ ]:
enc_bytes = syc.encrypt_message(
  sender_identity="alice@example.com",
  sender_keys=alice_keys,
  recipients=[syc.EncryptionRecipient("bob@example.com", bob_pub)],
  plaintext=b"hello bob!",
  filename_hint="msg.txt",
)
enc_bytes

b'SYC1\x01\r\x0c\x00\x00{"canon":"jcs-rfc8785","cipher":{"ciphertext_len":26,"last_segment_bytes":26,"nonce":"4wFzG1YAQkHeh9a08yUYvO0fS_ZEpMCx","segment_count":1,"suite":"xchacha20poly1305-v1"},"created_at":1763715856,"public_meta":{"filename_hint":"msg.txt"},"recipient_set_fpr":"953eb2f3524327d2a46cf3f743102412ee29032db6c31f4036487aac4e8f963b","recipients":[{"device_label":"default","identity":"bob@example.com","pqspk_fingerprint":"f2496c0f815b584e7af60df98b5ee3d1bce05c06166c3108177c55ec99ed7b5a","signed_prekey_id":1,"spk_fingerprint":"8eaf0f6c899711d6985d0fa79c1eda611a8d7cc20ef546c0f01857e1aba410b9"}],"sender":{"identity":"alice@example.com","ik_fingerprint":"0f604cf2a039ede8f1e85e8a4846e5f813deaa69d7e4c4ec8d1c11e3a63837d0"},"version":1,"wrappings":[{"device_label":"default","recipient_identity":"bob@example.com","wrap_ciphertext":"a1CbhRs9tf2KX_0MYre3p36o1cTljEK3N5oL-I0pFJU-uWf9lwORRVuB36ApUeyzdRHORz4yCX2wYZQ_rPYBy4MU8SLGYbx6CCI8CUEq8H8hoTXQa8QpOCeyz50WWYkt0_ajHHFBvL5IKTB5P2GPF1ZQJo

In [14]:
parsed = syc.parse_envelope(enc_bytes)
parsed

In [ ]:
syc.verify_envelope_signature(parsed, alice_pub.identity_key_bytes)

None


In [18]:
pt = syc.decrypt_message(
    recipient_identity="bob@example.com",
    recipient_keys=bob_keys,
    sender_bundle=alice_pub,
    envelope=parsed
)
print(f"decrypted message: {pt}")
assert pt == b"hello bob!"
print("encrypt/decrypt OK")

decrypted message: b'hello bob!'
encrypt/decrypt OK
